In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/03.silver-helpers"

In [0]:
%run "../0_common/01. env-config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))
display(results_df.head(5))

In [0]:
results_df = results_df.drop("url")

In [0]:
results_df = results_df.withColumnsRenamed(
    {
        "constructorId":"constructor_id",
        "driverId":"driver_id",
        "raceName":"race_name",
        "positionText":"finish_position_text",
        "date":"race_date",
        "grid":"grid_position",
        "laps":"completed_laps",
        "number":"car_number",
        "position":"finish_position"
    }
)

In [0]:
import pyspark.sql.functions as F
results_df = results_df.filter(
    F.col("season").isNotNull()&
    F.col("round").isNotNull()&
    F.col("driver_id").isNotNull()&
    F.col("constructor_id").isNotNull()
)

In [0]:
results_df = results_df.dropDuplicates(["season","round","driver_id","constructor_id"])

In [0]:
display(results_df.head(5))

In [0]:
results_df = results_df.withColumn(
    "race_name",
    F.initcap(F.col("race_name"))
)

In [0]:
columns_to_update = [col for col in results_df.columns if col not in ["season","round","driver_id","constructor_id"]]
columns_to_update

In [0]:
write_to_silver(
    df=results_df,
    target_table=silver_table,
    columns_to_update=columns_to_update,
    merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id"
)

In [0]:
display(spark.table(silver_table).limit(5))